- 8×A100 SXM4 + 6×NVSwitch 全互联平台
    - SXM4：GPU 模组形态（form factor），允许 400 W 功耗和 12 条 NVLink；不是服务器品牌。
    - “全互联”
        - 并不是 8 张 GPU 两两拉专线，那需要 $8 \times 7 / 2 = 28$ 组直接连接。
        - A100 GPU -> 6 × NVSwitch 组成的交换网络 -> 另一张 A100 GPU
            - 箭头1：12 × NVLink
            - 6 个 NVSwitch 中，每个交换机与每张 GPU 连接 2 条。所以每张 GPU 的连接数是 $6 \times 2 = 12$。
- 参考
    - https://github.com/NVIDIA/nvbandwidth.git
    - https://github.com/wilicc/gpu-burn.git
    - https://www.nvidia.com/en-us/data-center/a100/
    - https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/a100/pdf/PB-10577-001_v02.pdf

### 本地 & 远端

```
Host xxx
    Hostname ip
    User xxx
    ServerAliveInterval 30
    ServerAliveCountMax 3
```

- 本地：
    - 配置免密 ssh
        - `~/.ssh/config`
        - `ssh-copy-id -i ~/.ssh/id_ed25519.pub user@server`
            - 本机公钥会追加到 `/home/xx/.ssh/authorized_keys`
- 远端
    - 科学上网
        - `curl https://ifconfig.me/ip`：你看到我的公网来源 IP（public source IP）是什么？
    - bash -> zsh



```
# shell
echo $SHELL

sudo apt update
sudo apt install -y zsh
chsh -s "$(command -v zsh)"

# oh my zsh
sh -c "$(curl -fsSL https://raw.githubusercontent.com/ohmyzsh/ohmyzsh/master/tools/install.sh)"

sudo apt install -y zsh-autosuggestions zsh-syntax-highlighting
source /usr/share/zsh-autosuggestions/zsh-autosuggestions.zsh
source /usr/share/zsh-syntax-highlighting/zsh-syntax-highlighting.zsh

source ~/.zshrc
```

```
# proxy
proxy_off
printf 'direct: '
curl https://ifconfig.me/ip
printf '\n'

proxy_on
printf 'proxy:  '
curl https://ifconfig.me/ip
printf '\n'
```

- fzf
```
sudo apt update
sudo apt install -y fzf
```

### 基础配置

- linux 发行版：`cat /etc/os-release`
    - lsb_release -a
- 基本硬件信息查看

```
# memory
free -h
# 磁盘
df -hT

sudo apt update
sudo apt install -y inxi
sudo inxi -Fxxxz

sudo apt install neofetch
neofetch

sudo apt install btop
btop
```

- memory
    - `sudo dmidecode --type 17`
    - 16 × 64GiB DIMM = 1TiB
    - 32个内存槽中有16个已安装、16个为空。
```

sudo dmidecode --type 16 | grep -E 'Maximum Capacity|Number Of Devices'

sudo lshw -class memory -short

sudo lshw -class memory -short | grep "64GiB DIMM DDR4" | wc -l
sudo lshw -class memory -short | grep empty | wc -l
```

- nvme / ssd 及分区与挂载
    - SSD（与 HDD 相对）
        - ├─ SATA SSD
        - └─ NVMe SSD
    - 系统盘960GB级SATA SSD
    - 数据盘3.84TB级NVMe，挂载 /data1

```
# lsblk
/dev/sda
├── sda1  /boot/efi
└── sda2  /

/dev/nvme0n1
└── nvme0n1p1  /data1
```

- 网口

```
ip -br link
sudo ethtool enp113s0f0
```

#### 必要安装

```
sudo apt update
sudo apt install -y ripgrep

# 看网口网速
sudo apt install -y ifstat
```

### GPUs

- nvidia-smi
    - nvidia-smi topo -m
        - nv4/12: nv，nvlink，4/12，通道数
    - nvitop：基于 uv 安装
        - 安装 uv：`curl -LsSf https://astral.sh/uv/install.sh | sh`
            - `sudo apt update`
            - `sudo apt install -y curl`
            - 默认安装在 `~/.local/bin`，因此不需要 sudo
        - `uv tool install nvitop`
- gpu burn
    - volatile corrected/uncorrected ECC
    - -tc：using FLOATS, using Tensor Cores
    - -m: 默认 90% 的可用内存，(80714 MB available, using 72643 MB of it)
    - 输入矩阵 $A$ 和 $B$ 在初始化时生成一次。随后重复计算：
        - 8k=8192 的 shape，8192\*8192\*4=268435456 bytes=256MiB
        - thus performing 281 iterations：(72643 -2*256) / 256 = 281
            - $C_i=A\times B,\qquad i=0,\ldots,280$
            - 一个输入矩阵 A，输入矩阵 B，256MiB，多个结果矩阵 $C_0,C_1,\ldots$
        - 因为输入相同，理论上所有 $C_i$ 都应该相同。比较 kernel 会逐元素检查：
            - $|C_0[j]-C_i[j]|>0.001$
    - Gflop/s 如何计算
        - 对于一个 $N\times N$ 矩阵乘法：$C=A\times B$
        - 传统 FLOP 估算为：$2N^3$
        - 当 $N=8192$：2*8192^3 = 1.0995e+12 flop
            - 也就是一次 GEMM 大约 1.1 万亿次浮点运算。
            - $\text{GFLOP/s}=\frac{\text{时间段内完成的矩阵数}\times \text{OPS\_PER\_MUL}}{\text{时间秒数}\times10^9}$
        - 约 144–147 TFLOP/s/卡

```
./gpu_burn -l
# -m：默认也是 90% 的显存分配（分母是可用显存）
./gpu_burn -m 90% 3600
./gpu_burn -tc -m 90% 3600
```

| 数值类型 | 稠密峰值 | 2:4 稀疏峰值 |
|---|---:|---:|
| 普通 FP32 CUDA Core | **19.5 TFLOP/s** | 不适用 |
| TF32 Tensor Core | **156 TFLOP/s** | 312 TFLOP/s |
| FP16/BF16 Tensor Core | 312 TFLOP/s | 624 TFLOP/s |
| FP64 CUDA Core | 9.7 TFLOP/s | 不适用 |
| FP64 Tensor Core | 19.5 TFLOP/s | 不适用 |

#### cuda

```
ls -ld /usr/local/cuda*
/usr/local/cuda/bin/nvcc --version
```

### 数据路径

| 数据路径 | 使用的互联 |
|---|---|
| CPU 内存 ↔ A100 显存 | PCIe 4.0 ×16 |
| A100 ↔ A100 | NVLink/NVSwitch |
| A100 本地显存内部读写 | HBM2e |

- 主机到设备（host to device，即CPU及其内存传输到设备（GPU内存））
- 设备到主机（device to host，从设备（GPU内存）回传到主机（系统内存））
- 设备到设备（device to device，两个GPU之间直接传输数据的性能）
- gpu vram 到 gpu 计算核心：hbm2e
    - 2039 GB/s： (1.593*2)*5120 / 8
- Host↔GPU 走：
    - CPU DRAM → NUMA/PCIe Root → PEX88048 → PCIe Gen4 x16 → GPU HBM
    - 实际约 23–24 GB/s/方向。
- GPU↔GPU 走：
    - GPU HBM → 12×NVLink → 6×NVSwitch → 12×NVLink → GPU HBM
    - 不经过 CPU DRAM；单向任意 GPU 对达到 277–281 GB/s，约为理论单向 300 GB/s 的 92–94%。

### GPU （拓扑）连接与带宽

- PCIe Gen4 x16 理论为约 31.5 GB/s/方向；
```
nvidia-smi \
  --query-gpu=index,name,pci.bus_id,pcie.link.gen.current,pcie.link.gen.max,pcie.link.width.current,pcie.link.width.max \
  --format=csv

index, name, pci.bus_id, pcie.link.gen.current, pcie.link.gen.max, pcie.link.width.current, pcie.link.width.max
0, NVIDIA A100-SXM4-80GB, 00000000:0B:00.0, 4, 4, 16, 16
1, NVIDIA A100-SXM4-80GB, 00000000:10:00.0, 4, 4, 16, 16
2, NVIDIA A100-SXM4-80GB, 00000000:48:00.0, 4, 4, 16, 16
3, NVIDIA A100-SXM4-80GB, 00000000:4E:00.0, 4, 4, 16, 16
4, NVIDIA A100-SXM4-80GB, 00000000:9B:00.0, 4, 4, 16, 16
5, NVIDIA A100-SXM4-80GB, 00000000:A1:00.0, 4, 4, 16, 16
6, NVIDIA A100-SXM4-80GB, 00000000:DB:00.0, 4, 4, 16, 16
7, NVIDIA A100-SXM4-80GB, 00000000:E0:00.0, 4, 4, 16, 16
```

- nvidia-smi topo -m
    - 任意两卡之间均为 NV12
- nvidia-smi topo -p2p r 和 -p2p w 中所有 GPU 对均为：OK
- nvidia-smi nvlink --status 显示每张 GPU：Link 0–11: 25 GB/s
    - 每张 A100：$12\times25=300\ \text{GB/s/方向}$
    - 全双工理论值：$300+300=600\ \text{GB/s}$

- `build/nvbandwidth --list`
    - host_to_device_memcpy_ce
    - device_to_host_memcpy_ce
    - device_to_device_memcpy_write_ce
    - device_to_device_bidirectional_memcpy_write_ce
    - device_to_device_bidirectional_memcpy_write_sm
    - all_to_one_write_ce
    - one_to_all_write_ce
    - device_local_copy

```
# 24GB/s
build/nvbandwidth -t host_to_device_memcpy_ce -b 512 -i 3
# 23.5GB/s
build/nvbandwidth -t device_to_host_memcpy_ce -b 512 -i 3

# GPU↔GPU 单向测试
# 绝大多数：278–279 GB/s, 理论 300 GB/s
build/nvbandwidth -t device_to_device_memcpy_write_ce -b 512 -i 3


# GPU↔GPU 双向测试：SM 双向
# 323.05 GB/s ~ 397.88 GB/s
build/nvbandwidth -t device_to_device_bidirectional_memcpy_write_sm -b 512 -i 3
# Copy Engine 双向
# 434.61 GB/s ~ 534.30 GB/s，理论全双工：600GB/s
build/nvbandwidth -t device_to_device_bidirectional_memcpy_write_ce -b 512 -i 3


# NVSwitch 聚合入口和出口
# 7 GPU → 1 GPU
build/nvbandwidth -t all_to_one_write_ce -b 512 -i 3
# 1 GPU → 7 GPU
build/nvbandwidth -t one_to_all_write_ce -b 512 -i 3
# 7 个对端并不会给一张 GPU 提供 7*300=2100GB/s 每张 A100 总共只有 12 条 NVLink，因此源或目标 GPU 的注入/接收边界仍是：12*25=300GB/s

# GPU 内部 device-local copy
# 880 GB/s
build/nvbandwidth -t device_local_copy -b 512 -i 5
```

#### pcie_bw / hbm_bw

```
GPU=0 uv run \
  --no-project \
  --index https://download.pytorch.org/whl/cu130 \
  --with torch,numpy \
  python pcie_bw.py

uv run \
  --no-project \
  --index https://download.pytorch.org/whl/cu130 \
  --with torch,numpy \
  -- bash -c '
for gpu in {0..7}; do
    echo
    echo "===== GPU $gpu ====="
    GPU="$gpu" python pcie_bw.py
done
'
```

```
uv run --with numpy  --with torch hbm_bw.py
```

```
pcie_bw_core.py
Host pinned memory
        ↕
   PCIe DMA copy
        ↕
     GPU HBM

hbm_bw_core.py
GPU HBM
   ↕ read/write
Copy Engine or SM
   ↕
GPU HBM
```

| 脚本 | 数据路径 | 主要执行单元 | 理论上限 | 字节计数 |
|---|---|---|---:|---|
| `pcie_bw.py` | Host DRAM ↔ PCIe ↔ GPU HBM | GPU Copy Engine / DMA | PCIe Gen4 x16，约 31.5 GB/s/方向 | 只数传输 payload |
| `hbm_bw.py` copy | GPU HBM → GPU HBM | 本地 Copy Engine | HBM2e 2,039 GB/s | payload 与 HBM 读+写分别报告 |
| `hbm_bw.py` add | HBM → SM → HBM | SM/CUDA Core | HBM2e 2,039 GB/s | 2 次读 + 1 次写 |

| 项目 | PCIe 脚本 | HBM 脚本 |
|---|---|---|
| 是否跨 CPU/GPU 边界 | 是 | 否 |
| 是否经过 PCIe | 是 | 否 |
| Host NUMA 是否重要 | 非常重要 | 计时区间内不重要 |
| `copy_` 物理位置 | Host↔Device | Device↔Device |
| 理论峰值 | 约 31.5 GB/s/方向 | 2,039 GB/s |
| 本次实测 | 约 23.7 GB/s | 约 1,824 GB/s |

```python
x = torch.empty(
    NUM_ELEMENTS,
    dtype=torch.float32,
    device=device,
)

y = torch.empty_like(x)
z = torch.empty_like(x)

# Local copy：一读一写
# HBM source x
#    ↓ read N bytes
# GPU local copy
#    ↓ write N bytes
# HBM destination z
copy_seconds = measure_seconds(
    lambda: z.copy_(x)
)

# Stream add：两读一写
# z = x + y
# read x  = N bytes
# read y  = N bytes
# write z = N bytes
# total   = 3N bytes

# HBM -> L2/load units -> SM executes add -> store units -> HBM
add_seconds = measure_seconds(
    lambda: torch.add(x, y, out=z)
)
```

### nccl collective communication

- `git clone https://github.com/NVIDIA/nccl-tests.git`

### uv pytorch

```
uv run \
  --no-project \
  --index https://download.pytorch.org/whl/cu130 \
  --with torch \
  python -c '
import torch

print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
'
```